In [1]:
import getpass
import json
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

# Securely ingest your Anthropic Key as an in-memory string variable
anthropic_api_key = getpass.getpass("Enter your Anthropic API Key: ")

print("✅ Authentication initialization complete. Code execution locked in memory.")

✅ Authentication initialization complete. Code execution locked in memory.


In [2]:
# 1. DEFINE THE SHARED STATE GRAPH SCHEMA
class ReportState(TypedDict):
    raw_logs: str                 # The shared data input source
    executive_summary: str        # Target field updated by executive_writer
    developer_brief: str          # Target field updated by developer_writer

# 2. INITIALIZE THE CLAUDE MODEL CORE
# Setting temperature=0.3 provides a balance of professional narrative fluency and accuracy
llm_writer = ChatAnthropic(
    model="claude-sonnet-4-6", 
    temperature=0.3,
    anthropic_api_key=anthropic_api_key
)

# 3. CONSTRUCT THE RUNTIME NODES
def executive_writer_node(state: ReportState) -> dict:
    print("\n👔 [NODE -> EXECUTIVE WRITER]: Compiling corporate executive summaries...")
    
    system_instruction = (
        "You are an elite Corporate Communications Director. Your task is to take raw technical metrics "
        "and draft a high-level executive summary tailored exclusively for C-level management (CEO/CTO).\n\n"
        "Rules:\n"
        "1. Focus heavily on business metrics, operational risk assessment, and overall percentage success rates.\n"
        "2. Do NOT include raw code snippets, stack traces, or developer terms like 'NullPointerException'.\n"
        "3. Keep it professional, concise, and structured using clean business vocabulary."
    )
    
    user_prompt = f"Raw Technical Input Logs:\n{state['raw_logs']}"
    
    messages = [
        SystemMessage(content=system_instruction),
        HumanMessage(content=user_prompt)
    ]
    
    response = llm_writer.invoke(messages)
    return {"executive_summary": response.content}

def developer_writer_node(state: ReportState) -> dict:
    print("\n💻 [NODE -> DEVELOPER WRITER]: Extracting deep-dive engineering diagnostics...")
    
    system_instruction = (
        "You are a Principal Software SRE and Core Lead Engineer. Your task is to take raw technical logs "
        "and draft a highly detailed, aggressive development brief tailored for the software engineering team.\n\n"
        "Rules:\n"
        "1. Isolate the exact file systems, failure stack lines, database error tokens, and line anomalies.\n"
        "2. Omit high-level corporate fluff or business padding. Speak in direct technical engineering terms.\n"
        "3. Provide clear actionable bullet points stating what needs debugging immediately."
    )
    
    user_prompt = f"Raw Technical Input Logs:\n{state['raw_logs']}"
    
    messages = [
        SystemMessage(content=system_instruction),
        HumanMessage(content=user_prompt)
    ]
    
    response = llm_writer.invoke(messages)
    return {"developer_brief": response.content}

In [5]:
from langgraph.checkpoint.memory import MemorySaver

# 4. ASSEMBLE THE LANGGRAPH PARALLEL WORKFLOW Blueprint
workflow = StateGraph(ReportState)

# Register specialized writer nodes
workflow.add_node("executive_writer", executive_writer_node)
workflow.add_node("developer_writer", developer_writer_node)

# Map parallel layout (Fan-Out)
workflow.add_edge(START, "executive_writer")
workflow.add_edge(START, "developer_writer")

# Bridge endpoints to final step
workflow.add_edge("executive_writer", END)
workflow.add_edge("developer_writer", END)

# FIX: Instantiate an in-memory checkpointer ledger and pass it to compile
pipeline_memory = MemorySaver()
writer_agent_app = workflow.compile(checkpointer=pipeline_memory)

print("🎉 Parallel multi-audience graph compiled successfully with memory checkpointing enabled.")

🎉 Parallel multi-audience graph compiled successfully with memory checkpointing enabled.


In [6]:
# Raw, unformatted telemetry log payload
raw_telemetry_input = """
TIMESTAMP: 2026-06-15T20:10:04Z
TEST_RUN_ID: REGRESSION_BATCH_904
TOTAL_SUITES_EXECUTED: 150
PASSED_SUITES: 148
FAILED_SUITES: 2

--- CRITICAL EXCEPTION IN SUITE: PAYMENT_INTEGRATION ---
Thread: 0x4f2 - Fatal Crash
File: "/src/gateways/stripe_service.py", Line 114, in trigger_payment_intent
    raise NullPointerException("api_key reference resolved to null storage location")
ResponseCode: 500 Internal Error
Context: Customer checkout blocked for cart instances over $100.

--- WARNING IN SUITE: INVENTORY_RECONCILIATION ---
Latency spike detected: DB connection pool wait time exceeded 4500ms.
File: "/src/db/redis_cache.py", Line 42, in fetch_lock
Status: Auto-resolved via read-replica retry policies.
"""

initial_input = {
    "raw_logs": raw_telemetry_input,
    "executive_summary": "",
    "developer_brief": ""
}

# FIX: Define a tracking session configuration thread identifier
writer_config = {"configurable": {"thread_id": "parallel_writer_session_01"}}

print("🚀 Launching Parallel Multi-Audience Pipeline Application Run...")

# FIX: Pass the writer_config dictionary to the stream function
events = writer_agent_app.stream(initial_input, config=writer_config, stream_mode="updates")

for event in events:
    for node_name, state_update in event.items():
        print(f"\n🛑 [PAUSED] Parallel Node '{node_name}' successfully completed calculation step.")
        print(f"State Update Output Fragment Keys generated: {list(state_update.keys())}")
        
        action = input(f"\nPress Enter to ACKNOWLEDGE '{node_name}' payload and let the system continue (or 'quit'): ")
        if action.strip().lower() == "quit":
            print("❌ Loop interrupted by manual command flag.")
            break

# FIX: Pass the same thread configuration identifier directly to get_state
final_state = writer_agent_app.get_state(config=writer_config)
results_dict = final_state.values

print("\n" + "=" * 70)
print("👔 OUTPUT TARGET A: MANAGEMENT EXECUTIVE SUMMARY")
print("=" * 70)
print(results_dict.get("executive_summary", "No Summary Compiled."))

print("\n" + "=" * 70)
print("💻 OUTPUT TARGET B: CORE DEVELOPER DEEP-DIVE BRIEF")
print("=" * 70)
print(results_dict.get("developer_brief", "No Brief Compiled."))

🚀 Launching Parallel Multi-Audience Pipeline Application Run...

💻 [NODE -> DEVELOPER WRITER]: Extracting deep-dive engineering diagnostics...

👔 [NODE -> EXECUTIVE WRITER]: Compiling corporate executive summaries...

🛑 [PAUSED] Parallel Node 'executive_writer' successfully completed calculation step.
State Update Output Fragment Keys generated: ['executive_summary']

🛑 [PAUSED] Parallel Node 'developer_writer' successfully completed calculation step.
State Update Output Fragment Keys generated: ['developer_brief']

👔 OUTPUT TARGET A: MANAGEMENT EXECUTIVE SUMMARY
# Executive Summary — Regression Quality Assurance Review
**Report ID:** REGRESSION_BATCH_904 | **Date:** June 15, 2026 | **Classification:** C-Suite Briefing

---

## Overall System Health

| Metric | Value |
|---|---|
| Total Validation Suites Executed | 150 |
| Suites Passed | 148 |
| Suites Failed | 2 |
| **Overall Success Rate** | **98.7%** |

The platform demonstrated strong overall stability, with **98.7% of all validat